<a href="https://colab.research.google.com/github/imane-louzi/rag-finance-banque/blob/main/rag_banque_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline RAG - Projet Banque / Finance

Ce notebook construit un pipeline RAG (Retrieval-Augmented Generation) basique sur le dataset `sujet-ai/Sujet-Financial-RAG-EN-Dataset`.

Étapes :
1. Chargement des données
2. Préparation des chunks
3. Embeddings
4. Base vectorielle (FAISS)
5. Retrieval
6. Génération (LLM via OpenRouter)
7. Évaluation basique

A exécuter dans Google Colab.

## 0. Installation des dépendances

In [1]:
!pip install -q datasets sentence-transformers faiss-cpu openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 37.6 MB/s eta 0:00:00


## 1. Chargement des données

On charge le dataset directement depuis Hugging Face (pas besoin de téléchargement manuel).

In [2]:
from datasets import load_dataset

ds = load_dataset("sujet-ai/Sujet-Financial-RAG-EN-Dataset")
print(ds)

# On regarde la structure d'un exemple
print(ds["train"][0])

README.md:   0%|          | 0.00/7.84k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 10.3MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  916kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/98590 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7068 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'context'],
        num_rows: 98590
    })
    test: Dataset({
        features: ['question', 'context'],
        num_rows: 7068
    })
})
{'question': 'What is the purpose of Form 10-K as filed by Alphabet Inc. with the SEC?', 'context': 'UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________ FORM 10-K ___________________________________________ (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2023 OR ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from to . Commission file number: 001-37580 ___________________________________________ Alphabet Inc. (Exact name of registrant as specified in its charter) ___________________________________________ Delaware 61-1767919 (State or other jurisdiction of incorpora

In [3]:
# Conversion en pandas pour manipuler plus facilement
import pandas as pd

df = ds["train"].to_pandas()
print(df.columns.tolist())
print(df.shape)
df.head(3)

['question', 'context']
(98590, 2)


,question,context
0,What is the purpose of Form 10-K as filed by A...,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
1,What fiscal year does the annual report for Al...,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
2,Identify the two classes of securities registe...,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...


## 2. Préparation des chunks

On récupère les contextes uniques (colonne `context` ou équivalent selon le dataset). C'est notre base documentaire à indexer.

Adapte le nom de colonne ci-dessous selon ce que la cellule précédente t'a affiché (`df.columns.tolist()`).

In [4]:
# A adapter selon le nom exact de la colonne contexte affichée plus haut
context_col = "context"  # ou "Contexts" selon la version du dataset

chunks = df[context_col].drop_duplicates().reset_index(drop=True).tolist()
print(f"Nombre de chunks uniques : {len(chunks)}")
print(chunks[0][:500])

Nombre de chunks uniques : 4905
UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ___________________________________________ FORM 10-K ___________________________________________ (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2023 OR ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the transition period from to . Commission file number: 001-37580 ________________________


## 3. Embeddings

On utilise un modèle gratuit en local via `sentence-transformers`, pas besoin de clé API pour cette étape.

In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
print(chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/154 [00:00<?, ?it/s]

(4905, 384)


## 4. Base vectorielle (FAISS)

FAISS tourne entièrement en local, aucun serveur à installer.

In [6]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype(np.float32))
print(f"Index FAISS créé avec {index.ntotal} vecteurs")

Index FAISS créé avec 4905 vecteurs


## 5. Retrieval

Fonction qui prend une question et renvoie les k chunks les plus proches.

In [7]:
def retrieve(question, k=3):
    question_embedding = embedder.encode([question], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(question_embedding, k)
    return [chunks[i] for i in indices[0]]

# Test rapide
test_question = df.iloc[0]["question"] if "question" in df.columns else df.iloc[0]["Questions"]
results = retrieve(test_question)
for i, r in enumerate(results):
    print(f"--- Résultat {i+1} ---")
    print(r[:300])
    print()

--- Résultat 1 ---
3.Exhibits The documents listed in the Exhibit Index of this Annual Report on Form 10-K are incorporated by reference or are filed with this Annual Report on Form 10- K, in each case as indicated therein (numbered in accordance with Item 601 of Regulation S-K). ITEM 16. FORM 10-K SUMMARY None. 136

--- Résultat 2 ---
SIGNATURES Pursuant to the requirements of Section 13 or 15(d) of the Securities Exchange Act of 1934, the Registrant has duly caused this Annual Report on Form 10-K to be signed on its behalf by the undersigned, thereunto duly authorized. Date: January 30, 2024 ALPHABET INC. By: /S/ S UNDAR PICHAI 

--- Résultat 3 ---
and the RegistrantCurrent Report on Form 8-K (File No. 001-37580)October 2, 2015 10.05 u Director Arrangements Agreement, dated October 2, 2015, between Google Inc. and the RegistrantCurrent Report on Form 8-K (File No. 001-37580)October 2, 2015 10.06 u Alphabet Inc. Deferred Compensation Plan Curre



## 6. Génération

On construit un prompt avec la question + les chunks récupérés, envoyé à un LLM via OpenRouter.

Remplace `OPENROUTER_API_KEY` par ta clé (celle mentionnée avec le cours d'Andrew Ng).

In [13]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="sk",  # remplace par ta clé
)
def generate_answer(question, retrieved_chunks, model="openrouter/free"):
    context_text = "\n\n".join(retrieved_chunks)
    prompt = f"""Tu es un assistant spécialisé en analyse financière.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni ci-dessous.
Si le contexte ne permet pas de répondre, dis-le clairement.

Contexte :
{context_text}

Question : {question}

Réponse :"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

answer = generate_answer(test_question, results)
print(answer)

Le contexte indique que le Form 10-K d’Alphabet Inc. est un **rapport annuel déposé auprès de la SEC** afin de satisfaire aux exigences des **sections 13 ou 15(d) du Securities Exchange Act de 1934**. Il est signé par les dirigeants de la société et inclut ou incorpore par référence des documents listés dans l’Exhibit Index.

Le contexte ne donne pas davantage de détails sur le contenu complet ou les objectifs spécifiques du rapport.


## 7. Évaluation basique

On compare quelques réponses générées aux réponses de référence du dataset, sur un petit échantillon.

In [15]:
sample = df.sample(5, random_state=42)
question_col = "question"

for _, row in sample.iterrows():
    q = row[question_col]
    retrieved = retrieve(q)
    generated = generate_answer(q, retrieved)

    print(f"Question : {q}")
    print(f"Réponse générée : {generated}")
    print("=" * 80)

Question : What implications does the ability to increase the number of committee members have for the decision-making process within the Board of Directors?
Réponse générée : Le contexte ne fournit aucune information sur les conséquences spécifiques que l’augmentation du nombre de membres d’un comité pourrait avoir sur le processus de prise de décision au sein du conseil d’administration. Ainsi, il est impossible de répondre à la question à partir des éléments fournis.
Question : What percentage of data coverage is reported for the exposure to companies active in the fossil fuel sector?
Réponse générée : Le contexte fourni ne permet pas de répondre à cette question, car il ne contient aucune information sur le pourcentage de couverture des données relatif à l’exposition aux sociétés actives dans le secteur des combustibles fossiles.
Question : **Voting Procedures**: What is the voting requirement for decisions made at meetings of the Board of Directors when a quorum is present?
Répons

In [16]:
token_usage_log = []

def log_usage(step_name, response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return
    entry = {
        "step": step_name,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
    }
    token_usage_log.append(entry)
    print(f"[{step_name}] tokens utilisés : {entry['total_tokens']} "
          f"(prompt: {entry['prompt_tokens']}, réponse: {entry['completion_tokens']})")

def total_tokens_used():
    return sum(e["total_tokens"] for e in token_usage_log)

In [17]:
class ConversationMemory:
    def __init__(self, max_turns=5):
        self.history = []
        self.max_turns = max_turns

    def add(self, question, answer):
        self.history.append({"question": question, "answer": answer})
        self.history = self.history[-self.max_turns:]

    def as_context(self):
        if not self.history:
            return ""
        turns = [f"Q: {h['question']}\nR: {h['answer']}" for h in self.history]
        return "Historique de la conversation :\n" + "\n\n".join(turns)

memory = ConversationMemory()

In [18]:
def orchestrator_route(question, memory):
    context_hint = memory.as_context()
    prompt = f"""Tu es un routeur. Classe la question suivante dans une seule catégorie :
- "retrieval" si répondre nécessite de chercher dans des documents financiers externes
- "direct" si la question porte sur l'historique de la conversation, est une reformulation, ou peut être répondue sans nouvelle recherche documentaire

{context_hint}

Question : {question}

Réponds uniquement par un mot : retrieval ou direct."""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
    )
    log_usage("orchestrator", response)
    decision = response.choices[0].message.content.strip().lower()
    return "retrieval" if "retrieval" in decision else "direct"

In [19]:
def research_agent(question):
    retrieved_chunks = retrieve(question, k=3)
    context_text = "\n\n".join(retrieved_chunks)
    prompt = f"""Tu es un agent de recherche spécialisé en documents financiers.
Réponds à la question en te basant UNIQUEMENT sur le contexte fourni.
Si le contexte ne permet pas de répondre, dis-le clairement.

Contexte :
{context_text}

Question : {question}

Réponse :"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
    )
    log_usage("research_agent", response)
    return response.choices[0].message.content

In [20]:
def synthesis_agent(question, raw_answer, memory):
    context_hint = memory.as_context()
    prompt = f"""Tu es un agent de synthèse. Reformule la réponse brute ci-dessous de façon claire,
concise et bien structurée, en réponse à la question posée. Garde le sens exact, ne rajoute rien.

{context_hint}

Question : {question}
Réponse brute : {raw_answer}

Réponse finale :"""

    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
    )
    log_usage("synthesis_agent", response)
    return response.choices[0].message.content

In [21]:
def ask(question, memory=memory):
    route = orchestrator_route(question, memory)
    print(f"[Orchestrateur] route choisie : {route}")

    if route == "retrieval":
        raw_answer = research_agent(question)
    else:
        context_hint = memory.as_context()
        prompt = f"{context_hint}\n\nQuestion : {question}\n\nRéponds directement, sans inventer d'information nouvelle."
        response = client.chat.completions.create(
            model="openrouter/free",
            messages=[{"role": "user", "content": prompt}],
        )
        log_usage("direct_answer", response)
        raw_answer = response.choices[0].message.content

    final_answer = synthesis_agent(question, raw_answer, memory)
    memory.add(question, final_answer)
    return final_answer

q1 = df.iloc[0]["question"]
print("Question :", q1)
print("Réponse :", ask(q1))
print("\nTokens totaux utilisés jusqu'ici :", total_tokens_used())

Question : What is the purpose of Form 10-K as filed by Alphabet Inc. with the SEC?
[orchestrator] tokens utilisés : 955 (prompt: 112, réponse: 843)
[Orchestrateur] route choisie : direct
[direct_answer] tokens utilisés : 344 (prompt: 62, réponse: 282)
[synthesis_agent] tokens utilisés : 1215 (prompt: 187, réponse: 1028)
Réponse : **Objet :** Rapport annuel obligatoire pour les entreprises cotées aux États-Unis.

Le formulaire 10-K est le rapport annuel obligatoire déposé auprès de la SEC par les sociétés cotées aux États-Unis. Pour Alphabet Inc., il sert à présenter :

*   ses états financiers audités ;
*   son activité, ses risques et ses perspectives ;
*   la rémunération des dirigeants ;
*   la gouvernance d'entreprise ;
*   d'autres informations réglementaires.

Ce dépôt est effectué car Alphabet Inc. est une société anonyme cotée en bourse soumise au contrôle de la SEC.

Tokens totaux utilisés jusqu'ici : 2514


In [22]:
q2 = "Peux-tu reformuler ta réponse précédente en une phrase plus courte ?"
print("Question :", q2)
print("Réponse :", ask(q2))
print("\nTokens totaux utilisés jusqu'ici :", total_tokens_used())

Question : Peux-tu reformuler ta réponse précédente en une phrase plus courte ?
[orchestrator] tokens utilisés : 778 (prompt: 264, réponse: 514)
[Orchestrateur] route choisie : direct
[direct_answer] tokens utilisés : 369 (prompt: 250, réponse: 119)
[synthesis_agent] tokens utilisés : 529 (prompt: 285, réponse: 244)
Réponse : Le 10‑K est le rapport annuel d’Alphabet Inc. à la SEC, présentant ses états financiers audités, son activité, ses risques, sa gouvernance et d’autres informations réglementaires.

Tokens totaux utilisés jusqu'ici : 4190
